# マルチポートCLN：結合を含む回路へ

単一ポートの$V=ZI$を、複数ポートでは$\mathbf v=Z(s)\mathbf i$へ拡張します。対角成分だけでなく、他のポートへの相互作用である非対角成分が重要です。単一ポートCLNを複数並べるだけでは、一般に結合を再現できません。

行列Cauerでは抵抗・インダクタンスの行列を扱います。[既往の行列CLN](https://www.compumag.org/Proceedings/2019_Paris/files/papers/PB-A3-5.pdf)では複数電源による場を少数の回路要素で表す枠組みが示されています。

## まず守るべきもの：端子の向きと仕事

$\mathbf i=P\tilde{\mathbf i}$とポート座標を変えるとき、$\tilde{\mathbf v}=P^T\mathbf v$と組にします。すると$\mathbf v^T\mathbf i=\tilde{\mathbf v}^T\tilde{\mathbf i}$、$\tilde Z=P^TZP$です。可逆な座標変換と、情報を捨てるポート削減を区別します。

下は2ポートRLの最小例です。行列CLN生成の実装ではなく、結合・相反性・受動性・座標変換を確かめる検査例です。


In [1]:
import numpy as np
R=np.array([[2.,.4],[.4,1.]])
L=np.array([[.5,.1],[.1,.3]])
P=np.array([[1.,1.],[1.,-1.]])/np.sqrt(2)
v=np.array([2.,-.3]);inew=np.array([.2,.7]);i=P@inew
assert abs(v@i-(P.T@v)@inew)<1e-14
for w in (.1,1,10,100):
    Z=R+1j*w*L
    assert np.allclose(Z,Z.T) # reciprocity: transpose, not conjugate transpose
    hermitian=(Z+Z.conj().T)/2
    assert np.linalg.eigvalsh(hermitian).min()>0
    transformed=P.T@Z@P
    assert np.allclose(P.T@(Z@(P@inew)),transformed@inew)
    print(f'omega={w:g}: mutual Z12={Z[0,1]}, minimum dissipation eigenvalue={np.linalg.eigvalsh(hermitian).min():.6g}')
print('PASS: work preservation, reciprocity and positive dissipation')


omega=0.1: mutual Z12=(0.4+0.010000000000000002j), minimum dissipation eigenvalue=0.859688
omega=1: mutual Z12=(0.4+0.1j), minimum dissipation eigenvalue=0.859688
omega=10: mutual Z12=(0.4+1j), minimum dissipation eigenvalue=0.859688
omega=100: mutual Z12=(0.4+10j), minimum dissipation eigenvalue=0.859688
PASS: work preservation, reciprocity and positive dissipation


## 検査すべき量

| 対象 | 検査 |
|---|---|
| 相反な材料・境界の線形系 | $Z=Z^T$。複素共役転置との混同を避ける |
| 交流の散逸 | $(Z+Z^H)/2$の非負性 |
| 場との対応 | ポート仕事と損失・磁気エネルギーの収支 |
| ブロック再帰 | 行列の掛け順、階数落ち、励振の独立性 |
| ポート削減 | 消したポートの影響と、残した伝達特性の誤差 |

非線形FP-CLNで追加する高調波的な空間パターンを「ポート」として基底生成に使う場合、それは物理的な端子を増やすこととは限りません。表面モードの追加、体積基底の追加、実端子の追加を明示的に分けます。

このノートの再実行にはnumpyだけが必要です。実際の多導体系の行列CLNの公開用再現は別の段階です。[非線形と表面モード](03_nonlinear_fp_surface_modes.ipynb)と合わせて、その検証項目を整理しています。
